# Tutorial: Plan 004.6 Existing Read API Migration

**Audience:** Python users who want the complete public read API for current files, catalog-wide content, active tag vocabulary, and retained exact/history reads.

**Prerequisites:** Python 3.12+, an Archiver checkout, and familiarity with content identity (a SHA-256 identity for file bytes), current observations, and content-level tags.

This tutorial uses only disposable synthetic files and public Archiver APIs. Queries are immutable and lazy; their terminals read the catalog without changing source files. See the [Plan 004.6 implementation notes](../../docs/plans/004.6-existing-read-api-migration.md) and the [public query grammar](../../docs/query-grammar.md).

We will cover file details, catalog-wide content details, active tag vocabulary queries, retained exact/history methods, and concise CLI equivalents.

## Outline

1. Build a disposable catalog with duplicate, tagged, untagged, and historical-only content.
2. Add current-file tag details with complete reductions.
3. Query every persisted content identity and add root-scoped details.
4. Explore the active tag vocabulary with sorting, provenance, and regex predicates.
5. Use retained exact and historical reads, including explicit iterator cleanup.
6. Invoke the three migrated CLI workflows, complete an exercise, and clean up.

In [ ]:
from __future__ import annotations

from contextlib import closing
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, ContentRecord, CurrentFileTagView, TagProvenance
from archiver.cli import main
from archiver.query import content, path, tag_name, tag_usage, tags

## 1. Disposable catalog setup

The catalog is stored in the conventional in-root `.archiver` directory so the CLI can open the same database. The control directory is excluded from scans. We first scan seven source files, add content-level assertions, retract one assertion, remove one tagged path, and scan again. The removed bytes remain known content with historical observations and active tags, but no longer have a current path.

In [ ]:
temporary_directory = TemporaryDirectory(prefix="archiver-plan-0046-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
root.mkdir()
control_directory = root / ".archiver"
control_directory.mkdir()
database_path = control_directory / "catalog.sqlite"

seed_files = {
    "photos/copy.bin": b"photo",
    "photos/original.bin": b"photo",
    "notes.txt": b"notes",
    "draft.txt": b"draft",
    "orphan.bin": b"orphan",
    "untagged.bin": b"plain",
    "empty.bin": b"",
}
for relative_name, data in seed_files.items():
    destination = root / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)

catalog = Catalog.create(database_path)
first_summary = catalog.scan_directory(root, excluded_directories=(control_directory,))
first_summary

In [ ]:
shared_id = catalog.content_for_path(root, PurePosixPath("photos/copy.bin"))
notes_id = catalog.content_for_path(root, PurePosixPath("notes.txt"))
draft_id = catalog.content_for_path(root, PurePosixPath("draft.txt"))
orphan_id = catalog.content_for_path(root, PurePosixPath("orphan.bin"))
empty_id = catalog.content_for_path(root, PurePosixPath("empty.bin"))

user = TagProvenance("user", "tutorial-user", "1")
system = TagProvenance("system", "tutorial-classifier", "2")

for tag in ("photo", "reviewed"):
    catalog.add_content_tag(shared_id, tag, user)
catalog.add_content_tag(shared_id, "photo", system)
catalog.add_content_tag(notes_id, "document", system)
catalog.add_content_tag(notes_id, "reviewed", user)
catalog.add_content_tag(draft_id, "document", system)
catalog.add_content_tag(draft_id, "draft", user)
catalog.add_content_tag(orphan_id, "orphan", system)
catalog.add_content_tag(orphan_id, "retired", user)
catalog.retract_content_tag(orphan_id, "retired", user)
catalog.add_content_tag(empty_id, "empty", system)

# Removing a source path does not remove its content identity or tag history.
(root / "orphan.bin").unlink()
second_summary = catalog.scan_directory(root, excluded_directories=(control_directory,))

source_snapshot = {
    file_path.relative_to(root).as_posix(): (file_path.read_bytes(), file_path.stat().st_mtime_ns)
    for file_path in root.rglob("*")
    if file_path.is_file() and control_directory not in file_path.parents
}
{"first_scan_files": first_summary.files_observed, "current_scan_files": second_summary.files_observed}

## 2. Plan 004.6a — current-file tag details

`Catalog.files(root)` builds an immutable current-file query. `with_tags()` creates a detail view over the same bounded parent rows. The row bound affects `fetch()`, while the base query's `count()`, `distinct_content_count()`, and `total_size_bytes()` remain complete. Tag previews are alphabetical and may be bounded independently.

In [ ]:
photo_files = (
    catalog.files(root)
    .where(tags.all_of("photo"))
    .sort(path.asc())
    .limit(1)
)
tagged_preview = photo_files.with_tags(limit=2).fetch()
assert isinstance(tagged_preview[0], CurrentFileTagView)
assert tagged_preview[0].observation.relative_path == PurePosixPath("photos/copy.bin")
assert tagged_preview[0].active_tag_count == 2
assert tagged_preview[0].tags == ("photo", "reviewed")

photo_metrics = {
    "bounded_rows": len(tagged_preview),
    "file_count": photo_files.count(),
    "distinct_content_count": photo_files.distinct_content_count(),
    "total_size_bytes": photo_files.total_size_bytes(),
}
assert photo_metrics == {
    "bounded_rows": 1,
    "file_count": 2,
    "distinct_content_count": 1,
    "total_size_bytes": 10,
}
photo_metrics

Both duplicate paths share one content-level tag preview. A provenance restriction controls matching only; the displayed preview still includes every active tag on the selected content. `all_tags=True` removes the preview bound but does not change the parent row bound.

In [ ]:
system_photo = (
    catalog.files(root)
    .where(tags.all_of("photo", provenance="system"))
    .sort(path.asc())
    .limit(2)
)
system_photo_rows = system_photo.with_tags(limit=1).fetch()
assert [row.observation.relative_path.as_posix() for row in system_photo_rows] == [
    "photos/copy.bin",
    "photos/original.bin",
]
assert all(row.active_tag_count == 2 and row.tags == ("photo",) for row in system_photo_rows)
assert system_photo.with_tags(limit=1, all_tags=True).fetch()[0].tags == ("photo", "reviewed")

# The base query is reusable and unchanged by creating or evaluating its view.
assert photo_files.limit(5).count() == 2
{"restricted_match_paths": len(system_photo_rows), "unrestricted_preview": system_photo.with_tags(all_tags=True).fetch()[0].tags}

## 3. Plan 004.6b — catalog-wide content and details

`Catalog.contents()` starts from persisted content identities, not current paths. It therefore includes tagged and untagged content, current and historical-only content, and duplicate content only once. The default ordering is full identity order.

In [ ]:
all_contents = catalog.contents().fetch()
assert all(isinstance(record, ContentRecord) for record in all_contents)
assert len(all_contents) == 6
assert [(record.content_id.algorithm, record.content_id.digest) for record in all_contents] == sorted(
    (record.content_id.algorithm, record.content_id.digest) for record in all_contents
)

document_query = catalog.contents().where(tags.all_of("document"))
document_ids = {record.content_id for record in document_query.fetch()}
assert document_ids == {notes_id, draft_id}
assert catalog.contents().where(tags.all_of("orphan")).fetch()[0].content_id == orphan_id
assert catalog.contents().where(tags.all_of("missing")).fetch() == ()

assert {record.content_id for record in catalog.contents().where(tags.any_of("photo", "empty")).fetch()} == {
    shared_id,
    empty_id,
}
assert {record.content_id for record in catalog.contents().where(tags.all_of("document", provenance="system")).fetch()} == {
    notes_id,
    draft_id,
}

bounded_documents = document_query.limit(1)
assert len(bounded_documents.fetch()) == 1
assert bounded_documents.count() == 2
{"all_content_count": len(all_contents), "document_count": document_query.count()}

`with_details(root=...)` adds bounded current paths at one root and all-provenance active tag previews. The root scopes paths only; it does not remove catalog-wide content matches. An unknown but valid root consequently produces zero paths while retaining matching content rows.

In [ ]:
document_details = document_query.with_details(root=root, path_limit=1, tag_limit=1)
document_rows = document_details.fetch()
assert len(document_rows) == 2
assert document_details.current_path_count() == 2
assert all(row.current_path_count == 1 and len(row.current_paths) == 1 for row in document_rows)
assert all(row.active_tag_count == 2 and len(row.tags) == 1 for row in document_rows)

all_tag_rows = document_query.with_details(root=root, path_limit=2, tag_limit=1, all_tags=True).fetch()
assert {row.tags for row in all_tag_rows} == {("document", "reviewed"), ("document", "draft")}

unknown_root = workspace / "unknown-root"
unknown_root.mkdir()
unknown_details = document_query.with_details(root=unknown_root, path_limit=1, tag_limit=1)
assert unknown_details.current_path_count() == 0
assert len(unknown_details.fetch()) == 2
assert all(row.current_paths == () and row.current_path_count == 0 for row in unknown_details.fetch())
{"matching_contents": len(document_rows), "paths_at_unknown_root": unknown_details.current_path_count()}

## 4. Plan 004.6c — active tag vocabulary

`Catalog.tags()` describes active tag names and their usage. `count()` is complete even when `fetch()` is bounded. Sorting by usage is descending by default in these examples, with deterministic name-ascending ties; the library's default remains name ascending.

In [ ]:
vocabulary = catalog.tags()
default_usage = vocabulary.fetch()
assert [usage.tag for usage in default_usage] == ["document", "draft", "empty", "orphan", "photo", "reviewed"]
assert vocabulary.count() == 6

by_content = vocabulary.sort(tag_usage.content_count.desc()).fetch()
assert [usage.tag for usage in by_content[:2]] == ["document", "reviewed"]
by_assertions = vocabulary.sort(tag_usage.assertion_count.desc()).fetch()
assert [usage.tag for usage in by_assertions[:2]] == ["document", "photo"]
assert [usage.tag for usage in by_assertions[2:]] == ["reviewed", "draft", "empty", "orphan"]

{"default": [usage.tag for usage in default_usage], "most_assertions": [usage.tag for usage in by_assertions[:2]]}

In [ ]:
system_usage = catalog.tags().provenance("system").sort(tag_name.asc()).fetch()
user_usage = catalog.tags().provenance("user").sort(tag_name.asc()).fetch()
assert [usage.tag for usage in system_usage] == ["document", "empty", "orphan", "photo"]
assert [usage.tag for usage in user_usage] == ["draft", "photo", "reviewed"]
assert system_usage[0].system_assertion_count == 2
assert user_usage[1].user_assertion_count == 1

# Multiple regex clauses are ANDed, and matching follows Python re.search.
regex_query = (
    catalog.tags()
    .where(tag_name.regex("(?i)e"))
    .where(tag_name.regex("doc|review"))
)
assert [usage.tag for usage in regex_query.fetch()] == ["document", "reviewed"]
assert catalog.tags().where(tag_name.regex("retired")).count() == 0
assert catalog.tags().where(tag_name.regex("orphan")).count() == 1
{"system_tags": [usage.tag for usage in system_usage], "regex_matches": [usage.tag for usage in regex_query.fetch()]}

## 5. Plan 004.6d — retained exact and historical reads

The stable methods `current_scan`, `current_summary`, `content_for_path`, `tags_for_path`, `tags_for_content`, `scan_history`, and `observation_history` keep their existing signatures and result shapes. Their SQL now runs behind private compiler/executor helpers, but callers do not need a new public abstraction.

In [ ]:
current_scan = catalog.current_scan(root)
current_summary = catalog.current_summary(root)
assert current_scan is not None and current_scan.status == "completed"
assert current_summary is not None
assert current_summary.files_observed == 6
assert current_summary.distinct_content_count == 5
assert current_summary.duplicate_content_group_count == 1
assert catalog.content_for_path(root, PurePosixPath("photos/copy.bin")) == shared_id
assert {assertion.tag for assertion in catalog.tags_for_path(root, PurePosixPath("photos/original.bin"))} == {"photo", "reviewed"}
assert {assertion.tag for assertion in catalog.tags_for_content(orphan_id)} == {"orphan"}
{"current_scan_id": current_scan.id, "current_bytes": current_summary.total_bytes_observed}

In [ ]:
scan_history = tuple(catalog.scan_history())
observation_history = tuple(catalog.observation_history())
assert [scan.status for scan in scan_history] == ["completed", "completed"]
assert [scan.id for scan in scan_history] == sorted(scan.id for scan in scan_history)
assert len(observation_history) == 13
assert {observation.relative_path for observation in observation_history if observation.scan.id == scan_history[0].id} >= {
    PurePosixPath("orphan.bin"),
}
assert all(observation.scan.status == "completed" for observation in observation_history)
{"scan_ids": [scan.id for scan in scan_history], "historical_observations": len(observation_history)}

History iterators are lazy and own their database cursors. A plain `break` is not the cleanup contract; use `contextlib.closing` when a consumer intentionally stops early. Closing one history iterator does not make the catalog or an independent query unusable.

In [ ]:
partial_rows = []
with closing(catalog.observation_history()) as history:
    partial_rows.append(next(history))
assert partial_rows[0].scan.id == scan_history[0].id
assert catalog.tags().count() == 6
assert catalog.contents().where(content.is_(orphan_id)).count() == 1
"partial history iterator closed; catalog still usable"

## 6. Concise CLI parity

The typed Python API is the main teaching surface. These three calls exercise the migrated renderers once each: tag-aware current files, tag-filtered content details, and regex/sorted available tags. They all open the same in-root catalog and must return a successful exit code.

In [ ]:
assert main(["catalog", "files", str(root), "--tag", "photo", "--limit", "1", "--all-tags"]) == 0
assert main(["catalog", "tags", "find", str(root), "document", "--details", "--limit", "2", "--path-limit", "1"]) == 0
assert main(["catalog", "tags", "available", str(root), "--regex", "(?i)^(p|d)", "--sort", "assertions", "--limit", "3"]) == 0
"three migrated CLI workflows succeeded"

## Exercise

Build a provenance-restricted content query for identities having either `document` or `photo` from a system producer. Give it bounded current-path and tag details, then build a related system vocabulary query for four named tags. Before running the answer, predict the complete matching-content count, complete current-path count, and bounded vocabulary result.

In [ ]:
exercise_base = (
    catalog.contents()
    .where(tags.any_of("document", "photo", provenance="system"))
    .limit(1)
)
exercise_details = exercise_base.with_details(root=root, path_limit=1, tag_limit=2)
exercise_vocabulary = (
    catalog.tags()
    .provenance("system")
    .where(tag_name.regex("^(document|empty|orphan|photo)$"))
    .sort(tag_name.asc())
    .limit(3)
)
exercise_answer = {
    "displayed_content_rows": len(exercise_details.fetch()),
    "matching_contents": exercise_base.count(),
    "current_paths": exercise_details.current_path_count(),
    "vocabulary_preview": [usage.tag for usage in exercise_vocabulary.fetch()],
    "vocabulary_total": exercise_vocabulary.count(),
}
assert exercise_answer == {
    "displayed_content_rows": 1,
    "matching_contents": 3,
    "current_paths": 4,
    "vocabulary_preview": ["document", "empty", "orphan"],
    "vocabulary_total": 4,
}
exercise_answer

## Pitfalls and cleanup

- Bounds constrain displayed Python rows or previews; use complete reductions for totals.
- Tags describe content identity, so duplicate paths share a tag preview and historical-only content can still contribute vocabulary usage.
- A details root scopes only current paths; it does not scope catalog-wide content matching.
- Provenance predicates constrain matching or vocabulary scope, while detail previews show all active tags.
- Each terminal resolves its own current observation scope; separate terminals are not one public snapshot.
- Close partially consumed history iterators explicitly with `closing`.

The final assertion checks that every read demonstration and CLI query left source bytes and mtimes unchanged.

In [ ]:
source_after = {
    file_path.relative_to(root).as_posix(): (file_path.read_bytes(), file_path.stat().st_mtime_ns)
    for file_path in root.rglob("*")
    if file_path.is_file() and control_directory not in file_path.parents
}
assert source_after == source_snapshot
catalog.close()
temporary_directory.cleanup()
"read-only demonstrations complete; temporary catalog removed"